In [ ]:
!pip install -q opencv-python scikit-image scipy pandas tqdm


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import zipfile
import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm
from scipy.stats import skew, kurtosis

from skimage.feature import graycomatrix, graycoprops
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/balanced_dataset.zip"

EXTRACT_PATH = "/content/brain_mri_dataset"

os.makedirs(EXTRACT_PATH, exist_ok=True)

In [ ]:
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted successfully!")
print("Location:", EXTRACT_PATH)

Dataset extracted successfully!
Location: /content/brain_mri_dataset


In [ ]:
for root, dirs, files in os.walk(EXTRACT_PATH):
    level = root.replace(EXTRACT_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = ' ' * 2 * (level + 1)

    for file in files[:5]:
        print(f"{subindent}{file}")

    if level >= 2:
        dirs[:] = []

brain_mri_dataset/
  balanced_dataset/
    Training/
    Testing/


In [ ]:
CLASS_NAMES = {
    "glioma": 1,
    "meningioma": 1,
    "pituitary": 1,
    "notumor": 0
}

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

image_paths = []

for root, dirs, files in os.walk(EXTRACT_PATH):

    folder_name = os.path.basename(root).lower()

    if folder_name in CLASS_NAMES:

        label = CLASS_NAMES[folder_name]

        for file in files:

            if file.lower().endswith(IMAGE_EXTENSIONS):

                image_paths.append({
                    "path": os.path.join(root, file),
                    "class_name": folder_name,
                    "target": label
                })

print("Total images found:", len(image_paths))

print("\nClass distribution:")

class_counts = pd.Series(
    [x["class_name"] for x in image_paths]
).value_counts()

print(class_counts)

Total images found: 11264

Class distribution:
notumor       5864
meningioma    1800
pituitary     1800
glioma        1800
Name: count, dtype: int64


In [ ]:
FEATURE_NAMES = [
    "Contrast",
    "Dissimilarity",
    "Standard Deviation",
    "Variance",
    "Mean",
    "Homogeneity",
    "Entropy",
    "Kurtosis",
    "Correlation",
    "Skewness",
    "Energy",
    "ASM",
    "Coarseness",
    "PSNR",
    "SSIM",
    "MSE",
    "DC"
]

print("Number of features:", len(FEATURE_NAMES))

Number of features: 17


In [ ]:
IMG_SIZE = (224, 224)

def preprocess_image(image_path):

    image = cv2.imread(image_path)

    if image is None:
        return None

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    gray = cv2.resize(
        gray,
        IMG_SIZE,
        interpolation=cv2.INTER_AREA
    )

    return gray

In [ ]:
def extract_features(image_path):

    gray = preprocess_image(image_path)

    if gray is None:
        return None


    image_float = gray.astype(np.float64)


    mean_value = np.mean(image_float)


    variance_value = np.var(image_float)


    std_value = np.std(image_float)


    skewness_value = skew(
        image_float.flatten(),
        bias=False
    )


    kurtosis_value = kurtosis(
        image_float.flatten(),
        bias=False
    )


    histogram = cv2.calcHist(
        [gray],
        [0],
        None,
        [256],
        [0, 256]
    ).flatten()

    probability = histogram / (histogram.sum() + 1e-12)

    probability = probability[probability > 0]

    entropy_value = -np.sum(
        probability * np.log2(probability)
    )




    gray_32 = (gray / 8).astype(np.uint8)

    glcm = graycomatrix(
        gray_32,
        distances=[1],
        angles=[0],
        levels=32,
        symmetric=True,
        normed=True
    )


    contrast_value = graycoprops(
        glcm,
        "contrast"
    )[0, 0]


    dissimilarity_value = graycoprops(
        glcm,
        "dissimilarity"
    )[0, 0]


    homogeneity_value = graycoprops(
        glcm,
        "homogeneity"
    )[0, 0]


    correlation_value = graycoprops(
        glcm,
        "correlation"
    )[0, 0]


    asm_value = graycoprops(
        glcm,
        "ASM"
    )[0, 0]


    energy_value = graycoprops(
        glcm,
        "energy"
    )[0, 0]



    kernel_size = 5

    local_mean = cv2.blur(
        image_float,
        (kernel_size, kernel_size)
    )

    local_variance = cv2.blur(
        image_float ** 2,
        (kernel_size, kernel_size)
    ) - local_mean ** 2

    coarseness_value = np.mean(
        np.sqrt(np.maximum(local_variance, 0))
    )



    blurred = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    mse_value = np.mean(
        (image_float - blurred.astype(np.float64)) ** 2
    )


    if mse_value == 0:
        psnr_value = float("inf")
    else:
        psnr_value = peak_signal_noise_ratio(
            gray,
            blurred,
            data_range=255
        )


    ssim_value = structural_similarity(
        gray,
        blurred,
        data_range=255
    )




    dc_value = np.abs(
        np.fft.fft2(image_float)[0, 0]
    ) / (image_float.size * 255.0)



    features = [
        contrast_value,
        dissimilarity_value,
        std_value,
        variance_value,
        mean_value,
        homogeneity_value,
        entropy_value,
        kurtosis_value,
        correlation_value,
        skewness_value,
        energy_value,
        asm_value,
        coarseness_value,
        psnr_value,
        ssim_value,
        mse_value,
        dc_value
    ]

    return features

In [ ]:
test_image = image_paths[0]["path"]

features = extract_features(test_image)

print("Image:", test_image)
print("Number of extracted features:", len(features))

for name, value in zip(FEATURE_NAMES, features):
    print(f"{name}: {value}")

Image: /content/brain_mri_dataset/balanced_dataset/Training/notumor/Tr-no_0206.jpg
Number of extracted features: 17
Contrast: 0.8233103779628445
Dissimilarity: 0.3498558616271621
Standard Deviation: 45.87673614507718
Variance: 2104.674919325031
Mean: 41.30239556760204
Homogeneity: 0.8623231732684029
Entropy: 5.475322246551514
Kurtosis: -1.301911298692166
Correlation: 0.9869543660336982
Skewness: 0.5985328986728086
Energy: 0.4244973742313425
ASM: 0.1801980207293044
Coarseness: 5.323249666083953
PSNR: 37.4307490726476
SSIM: 0.9711773297712196
MSE: 11.749123086734693
DC: 0.1619701786964786


In [ ]:
all_data = []

failed_images = []

for item in tqdm(
    image_paths,
    desc="Extracting MRI features"
):

    try:

        features = extract_features(
            item["path"]
        )

        if features is None:
            failed_images.append(item["path"])
            continue


        row = dict(
            zip(FEATURE_NAMES, features)
        )

        row["Image"] = os.path.basename(item["path"])


        row["Class"] = item["class_name"]


        row["Target"] = item["target"]

        all_data.append(row)

    except Exception as e:

        failed_images.append(
            (item["path"], str(e))
        )

print("Extraction completed!")
print("Successful images:", len(all_data))
print("Failed images:", len(failed_images))

Extracting MRI features: 100%|██████████| 11264/11264 [02:33<00:00, 73.44it/s]

Extraction completed!
Successful images: 11264
Failed images: 0


In [ ]:
df = pd.DataFrame(all_data)

CSV_PATH = "/content/drive/MyDrive/bd.csv"

df.to_csv(
    CSV_PATH,
    index=False
)

print("CSV saved:", CSV_PATH)

CSV saved: /content/drive/MyDrive/bd.csv


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/bd.csv")

print(df.columns.tolist())

['Contrast', 'Dissimilarity', 'Standard Deviation', 'Variance', 'Mean', 'Homogeneity', 'Entropy', 'Kurtosis', 'Correlation', 'Skewness', 'Energy', 'ASM', 'Coarseness', 'PSNR', 'SSIM', 'MSE', 'DC', 'Image', 'Class', 'Target']


In [ ]:
print(df["Target"].value_counts())

print("\nTarget meaning:")
print("0 = No Tumor")
print("1 = Tumor")

Target
0    5864
1    5400
Name: count, dtype: int64

Target meaning:
0 = No Tumor
1 = Tumor


In [ ]:
print("\nOriginal classes:")
print(df["Class"].value_counts())


Original classes:
Class
notumor       5864
meningioma    1800
pituitary     1800
glioma        1800
Name: count, dtype: int64


In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nInfinite values:")

numeric_columns = FEATURE_NAMES

print(
    np.isinf(
        df[numeric_columns].to_numpy()
    ).sum()
)

Missing values:
Contrast              0
Dissimilarity         0
Standard Deviation    0
Variance              0
Mean                  0
Homogeneity           0
Entropy               0
Kurtosis              0
Correlation           0
Skewness              0
Energy                0
ASM                   0
Coarseness            0
PSNR                  0
SSIM                  0
MSE                   0
DC                    0
Image                 0
Class                 0
Target                0
dtype: int64

Infinite values:
0


In [ ]:
df[FEATURE_NAMES] = df[FEATURE_NAMES].replace(
    [np.inf, -np.inf],
    np.nan
)

print(
    "Total missing values after infinity handling:",
    df[FEATURE_NAMES].isnull().sum().sum()
)

Total missing values after infinity handling: 0


In [ ]:
df = df.dropna(
    subset=FEATURE_NAMES
).reset_index(drop=True)

print("Final dataset shape:", df.shape)

Final dataset shape: (11264, 20)


In [ ]:
CSV_PATH = "/content/drive/MyDrive/bd.csv"

df.to_csv(
    CSV_PATH,
    index=False
)

print("CSV saved successfully!")
print(CSV_PATH)

CSV saved successfully!
/content/drive/MyDrive/bd.csv


In [ ]:
loaded_df = pd.read_csv(CSV_PATH)

print("CSV shape:", loaded_df.shape)

print("\nColumns:")
print(loaded_df.columns.tolist())

print("\nFirst 5 rows:")
display(loaded_df.head())

CSV shape: (11264, 20)

Columns:
['Contrast', 'Dissimilarity', 'Standard Deviation', 'Variance', 'Mean', 'Homogeneity', 'Entropy', 'Kurtosis', 'Correlation', 'Skewness', 'Energy', 'ASM', 'Coarseness', 'PSNR', 'SSIM', 'MSE', 'DC', 'Image', 'Class', 'Target']

First 5 rows:


,Contrast,Dissimilarity,Standard Deviation,Variance,Mean,Homogeneity,Entropy,Kurtosis,Correlation,Skewness,Energy,ASM,Coarseness,PSNR,SSIM,MSE,DC,Image,Class,Target
0,0.823310,0.349856,45.876736,2104.674919,41.302396,0.862323,5.475322,-1.301911,0.986954,0.598533,0.424497,0.180198,5.323250,37.430749,0.971177,11.749123,0.161970,Tr-no_0206.jpg,notumor,0
1,5.320227,0.916360,58.855533,3463.973779,46.346799,0.765500,5.871881,0.509064,0.949824,1.143142,0.512031,0.262176,12.181504,27.785269,0.938722,108.281290,0.181752,Tr-no_1198.jpg,notumor,0
2,6.833480,1.277306,76.699175,5882.763373,82.533701,0.660675,7.025455,-0.949339,0.962859,0.510292,0.287774,0.082814,17.745696,27.314099,0.922799,120.689772,0.323662,Tr-no_230.jpg,notumor,0
3,4.513693,0.787036,57.166259,3267.981192,83.894571,0.793339,4.417641,-0.757500,0.953668,0.799867,0.548323,0.300658,8.955850,28.017635,0.928087,102.640047,0.328998,N_40_BR_.jpg,notumor,0
4,5.477258,0.948390,58.010721,3365.243725,36.646225,0.763588,4.378100,1.050869,0.945841,1.460866,0.598739,0.358489,13.305476,27.059114,0.952019,127.987923,0.143711,Tr-no_962.jpg,notumor,0


In [ ]:
print("=" * 60)
print("BRAIN MRI FEATURE DATASET")
print("=" * 60)

print("Total samples:", len(df))

print("\n17 Features:")
for i, feature in enumerate(FEATURE_NAMES, 1):
    print(f"{i:2d}. {feature}")

print("\nTarget:")
print("0 = No Tumor")
print("1 = Tumor")

print("\nBinary distribution:")
print(df["Target"].value_counts())

print("\nOriginal class distribution:")
print(df["Class"].value_counts())

print("\nCSV location:")
print(CSV_PATH)

BRAIN MRI FEATURE DATASET
Total samples: 11264

17 Features:
 1. Contrast
 2. Dissimilarity
 3. Standard Deviation
 4. Variance
 5. Mean
 6. Homogeneity
 7. Entropy
 8. Kurtosis
 9. Correlation
10. Skewness
11. Energy
12. ASM
13. Coarseness
14. PSNR
15. SSIM
16. MSE
17. DC

Target:
0 = No Tumor
1 = Tumor

Binary distribution:
Target
0    5864
1    5400
Name: count, dtype: int64

Original class distribution:
Class
notumor       5864
meningioma    1800
pituitary     1800
glioma        1800
Name: count, dtype: int64

CSV location:
/content/drive/MyDrive/bd.csv


In [ ]:
row = dict(
    zip(FEATURE_NAMES, features)
)


row["Image"] = os.path.basename(item["path"])

row["Class"] = item["class_name"]


row["Target"] = item["target"]

all_data.append(row)

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/bd.csv")

print(df.columns.tolist())
print(df.shape)
print(df["Target"].value_counts())

['Contrast', 'Dissimilarity', 'Standard Deviation', 'Variance', 'Mean', 'Homogeneity', 'Entropy', 'Kurtosis', 'Correlation', 'Skewness', 'Energy', 'ASM', 'Coarseness', 'PSNR', 'SSIM', 'MSE', 'DC', 'Image', 'Class', 'Target']
(11264, 20)
Target
0    5864
1    5400
Name: count, dtype: int64
